# Colab Runner (Worker GPU A100) — `parakeet-ptbr-multitalker`

Este notebook segue a arquitetura **Spec-Driven Development (SDD)** (`Spec 00` e `Spec 04`).
- **Não escreva lógica de negócio diretamente nas células.**
- As 4 células abaixo apenas sincronizam o repositório Git, validam os contratos com `pytest`, extraem os dados compactados do Google Drive para o SSD NVMe local (`/content/local_dataset`) e disparam o `src.trainer` com checkpoints persistentes no Drive.

In [ ]:
# CÉLULA 1 — Montar Google Drive, Clonar Repositório e Instalar Dependências
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/gabrielmedeirossouza/multitalker-model.git"
!git clone {REPO_URL} /content/multitalker-model || (cd /content/multitalker-model && git pull)
%cd /content/multitalker-model

!pip install -q Cython packaging
!pip install -q -e ".[dev,colab]"

In [ ]:
# CÉLULA 2 — Sincronizar Código/Estado do Drive e Rodar Gatekeeper SDD
!git pull
!python -m src.state sync-from-drive --drive_state_dir /content/drive/MyDrive/parakeet_ptbr/state || true
!pytest tests/ -v

In [ ]:
# CÉLULA 3 — Garantir Dataset e Tokenizador no SSD Local da VM (Gera nas Steps 01-04 ou Extrai do Drive)
!python -m src.data_prep --ensure_colab_dataset \
    --dataset fleurs_pt \
    --local_dataset_dir /content/local_dataset \
    --drive_root_dir /content/drive/MyDrive/parakeet_ptbr \
    --update_state
!ls -lh /content/local_dataset

In [ ]:
# CÉLULA 4 — Disparar Treinamento (Fase 1: Warm-up -> Fase 2: Multitalker) com Checkpoints no Drive
# Caso a sessão do Colab caia, basta reexecutar: o exp_manager retomará do último checkpoint (resume_if_exists=True).

STAGE = "warmup"  # Altere para "multitalker" após concluir as 5 épocas de warm-up
MANIFEST = "/content/local_dataset/clean_train_manifest.json" if STAGE == "warmup" else "/content/local_dataset/multitalker_train_manifest.json"
VAL_MANIFEST = "/content/local_dataset/clean_val_manifest.json" if STAGE == "warmup" else "/content/local_dataset/multitalker_val_manifest.json"
CHECKPOINT_DIR = f"/content/drive/MyDrive/parakeet_checkpoints/exp_v1_{STAGE}"
TOKENIZER_DIR = "/content/drive/MyDrive/parakeet_ptbr/tokenizer_ptbr_1024"

!python -m src.trainer \
    --stage {STAGE} \
    --manifest {MANIFEST} \
    --val_manifest {VAL_MANIFEST} \
    --checkpoint_dir {CHECKPOINT_DIR} \
    --tokenizer_dir {TOKENIZER_DIR} \
    --update_state